# Week 3: AI Agents with Grounding Tools

Welcome to week 3 of April's Developer Challenge on AI! This week you will combine what you learned in weeks 1 and 2 to build a more powerful AI agent that uses the grounding service as a tool.

## What You'll Build

In this exercise, you'll enhance the Hamburg Social Welfare Case Manager agent from Week 2 by equipping it with a **custom grounding tool**. This allows the agent to:
- Query the vector database on demand for specific information
- Retrieve Hamburg-specific social services documentation
- Make decisions based on up-to-date, grounded knowledge
- Provide more accurate and verifiable responses

## The Power of Tool-Equipped Agents

Unlike Week 2 where the agent relied solely on its base model knowledge, your Week 3 agent will:
- **Autonomously decide** when to retrieve external information
- **Access real documentation** instead of relying on training data
- **Provide traceable answers** backed by actual source documents
- **Stay current** with information from the knowledge base

## What You'll Learn

By the end of this exercise, you'll understand how to:
- Create custom tools that wrap external services
- Enable agents to use tools based on query requirements
- Integrate RAG capabilities with agentic workflows
- Build end-to-end solutions with autonomous knowledge retrieval

This is where AI agents truly shine - combining reasoning capabilities with the ability to access and utilize external knowledge sources!

Let's get started!

### Assign the service key information to environmental variables

In [ ]:
"""Run in trial environment with config.json file from the technical academy"""
import os
import json

home_dir = os.path.expanduser('~')
aicore_dir = os.path.join(home_dir, '.aicore')
os.makedirs(aicore_dir, exist_ok=True)

config_path = os.path.join(aicore_dir, 'config.json')
with open(config_path, 'r') as config_file:
    config_data = json.load(config_file)

    os.environ["AICORE_AUTH_URL"]=config_data["AICORE_AUTH_URL"]+"/oauth/token"
    os.environ["AICORE_CLIENT_ID"]=config_data["AICORE_CLIENT_ID"]
    os.environ["AICORE_CLIENT_SECRET"]=config_data["AICORE_CLIENT_SECRET"]
    os.environ["AICORE_BASE_URL"]=config_data["AICORE_BASE_URL"]

    os.environ["LITELLM_PROVIDER"]= "sap"
    os.environ["AICORE_RESOURCE_GROUP"]="doc-grounding"

DATA_REPOSITORY_ID = "d7834fe4-83f0-4d6e-8583-1be833a2ca0b"

"""Run locally with .env file"""
# from dotenv import load_dotenv
# load_dotenv()

# AICORE_ORCHESTRATION_DEPLOYMENT_URL = "https://api.ai.prod.eu-central-1.aws.ml.hana.ondemand.com/v2/inference/deployments/D276250e074fc028/v2"

### Import Cloud SDK for AI - Retrieval API libraries

In [ ]:
import json

from gen_ai_hub.document_grounding.client import RetrievalAPIClient
from gen_ai_hub.document_grounding.models.retrieval import (
    RetrievalSearchInput,
    RetrievalSearchFilter,
)
from gen_ai_hub.orchestration.models.document_grounding import DataRepositoryType

from crewai import Agent, Crew, Task
from crewai.tools import tool

### Built a Tool for the Agent

In [ ]:
@tool("call_grounding_service")
def call_grounding_service(user_question: str) -> str:
    """Function to call the grounding service and retrieve relevant information based on the user's question."""
    retrieval_client = RetrievalAPIClient()

    search_filter = RetrievalSearchFilter(
        id="vector",
        dataRepositoryType=DataRepositoryType.VECTOR.value,
        dataRepositories=[DATA_REPOSITORY_ID],
        searchConfiguration={
            "maxChunkCount": 5
        },
    )

    search_input = RetrievalSearchInput(
        query=user_question,
        filters=[search_filter],
    )

    response = retrieval_client.search(search_input)

    response_dict = json.dumps(response.model_dump(), indent=2)
    return response_dict

### Create the Agent and Give it Access to the Tool

In [ ]:
# Create a Hamburg Social Welfare Case Manager Agent
welfare_agent = Agent(
    role="Hamburg Social Welfare Case Manager",
    goal="Process and manage social welfare applications for the city of Hamburg, matching citizens with appropriate social services and benefits based on their needs and circumstances. Use the call_grounding_service tool to retrieve relevant information from Hamburg's social services knowledge base to inform your assessments and recommendations.",
    backstory="You are an experienced social welfare case manager working for the city of Hamburg. You have deep knowledge of Hamburg's social services programs, eligibility requirements, local resources, and citizen support initiatives. Your role is to help vulnerable populations access the social welfare benefits they need.",
    llm="sap/gpt-4o",  # provider/llm - Using one of the models from SAP's model library in Generative AI Hub
    tools=[call_grounding_service],
    verbose=True
)

# Get user input
print("Hamburg Social Welfare Case Management System")
print("=" * 50)
user_question = input("\nPlease describe your social welfare inquiry or situation:\n> ")

# Create a task for the welfare case manager with user input
process_welfare_task = Task(
    description=f"Process the following social welfare inquiry for a Hamburg citizen: {user_question}\n\nBased on this inquiry and the information from the grounding service, determine eligibility for social welfare programs and recommend appropriate services.",
    expected_output="Structured social welfare assessment with eligibility determination, service recommendations, and personalized support plan for Hamburg citizens.",
    agent=welfare_agent
)

# Create a crew with the welfare agent
crew = Crew(
    agents=[welfare_agent],
    tasks=[process_welfare_task],
    verbose=True
)

# Execute the crew
if __name__ == "__main__":
    result = crew.kickoff()
    print("\n" + "="*50)
    print("Hamburg Social Welfare Assessment Report:")
    print("="*50)
    print(result)

Please submit a screenshot of the LLM's response directly under the [Week 3 Post - TBD]()